# 01 Collect Daily Panel

Pulls public sources from the submitted synopsis: **FRED, EIA (optional), GDELT, PortWatch (when CSV available)**.

Writes:
- `data/raw/*.csv`
- `data/processed/daily_panel.csv`
- `data/raw/collection_meta.json`

No Kaggle. No invented market history.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from project_paths import load_env, ensure_output_dirs
ROOT = load_env()
PATHS = ensure_output_dirs(ROOT)
print("ROOT", ROOT)
print({k: str(v) for k, v in PATHS.items()})


ROOT /Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold
{'root': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold', 'raw': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/data/raw', 'processed': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/data/processed', 'outputs': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/notebooks/outputs/latest', 'figures': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/notebooks/outputs/latest/figures', 'tables': '/Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/notebooks/outputs/latest/tables'}


In [2]:
from fetch_panel import build_daily_panel
import pandas as pd
import json

panel, meta = build_daily_panel()
display(panel.tail())
print(panel.shape)
print(json.dumps(meta, indent=2))
panel.head(3)


FRED_API_KEY missing; using public FRED CSV downloads


GDELT TimelineVol non-JSON/status 429


GDELT TimelineTone non-JSON/status 429


Wrote /Users/govind/Documents/govind/DBA/QM_640_Analytics_Capstone/github_scaffold/data/processed/daily_panel.csv rows= 2113
Meta: {'failures': ['EIA unavailable or key missing', 'PortWatch CSV not downloadable yet; physical IVs use observed market stress stand-ins documented in physical_iv_note (replace when PortWatch file is added to data/raw)'], 'sources_ok': ['FRED', 'GDELT'], 'portwatch': 'pending_real_csv'}


,date,wti_spot,brent_spot,natgas_hh,wheat,corn,soy,gdelt_tone,commodity_roll_yield,grain_calendar_spread,...,source_notes,bottleneck_index_5d,port_congestion_hours,chokepoint_vessel_count,vessel_draught_anomaly,physical_iv_note,volatility_regime,omega_ratio_proxy,news_event_impact_score,finbert_risk_probability
2108,2026-10-10,98.73,130.03,3.10,228.738845,213.190154,426.328157,NaN,-2.71,15.548691,...,portwatch_pending,NaN,0.010540,0.970918,1.164183,port_congestion_hours and chokepoint intensity...,1,0.984727,NaN,0.5
2109,2026-10-11,98.73,130.03,3.10,228.738845,213.190154,426.328157,NaN,1.76,15.548691,...,portwatch_pending,NaN,0.010312,0.949837,1.140315,port_congestion_hours and chokepoint intensity...,0,1.173322,NaN,0.5
2110,2026-10-12,98.73,129.95,3.10,228.738845,213.190154,426.328157,NaN,2.32,15.548691,...,portwatch_pending,NaN,0.007776,0.716299,1.116096,port_congestion_hours and chokepoint intensity...,0,1.203530,NaN,0.5
2111,2026-10-13,100.86,125.78,3.14,228.738845,213.190154,426.328157,NaN,7.48,15.548691,...,portwatch_pending,NaN,0.005271,0.485571,1.370205,port_congestion_hours and chokepoint intensity...,0,1.511489,NaN,0.5
2112,2026-10-14,101.91,126.55,3.22,228.738845,213.190154,426.328157,NaN,6.03,15.548691,...,portwatch_pending,NaN,0.006397,0.589240,1.476874,port_congestion_hours and chokepoint intensity...,0,1.426660,NaN,0.5


(2113, 21)
{
  "failures": [
    "EIA unavailable or key missing",
    "PortWatch CSV not downloadable yet; physical IVs use observed market stress stand-ins documented in physical_iv_note (replace when PortWatch file is added to data/raw)"
  ],
  "sources_ok": [
    "FRED",
    "GDELT"
  ],
  "portwatch": "pending_real_csv"
}


,date,wti_spot,brent_spot,natgas_hh,wheat,corn,soy,gdelt_tone,commodity_roll_yield,grain_calendar_spread,...,source_notes,bottleneck_index_5d,port_congestion_hours,chokepoint_vessel_count,vessel_draught_anomaly,physical_iv_note,volatility_regime,omega_ratio_proxy,news_event_impact_score,finbert_risk_probability
0,2021-01-01,NaN,NaN,NaN,237.94425,231.754461,510.669604,NaN,NaN,6.189788,...,portwatch_pending,NaN,NaN,NaN,NaN,port_congestion_hours and chokepoint intensity...,0,NaN,NaN,0.5
1,2021-01-02,NaN,NaN,NaN,237.94425,231.754461,510.669604,NaN,NaN,6.189788,...,portwatch_pending,NaN,NaN,NaN,NaN,port_congestion_hours and chokepoint intensity...,0,NaN,NaN,0.5
2,2021-01-03,NaN,NaN,NaN,237.94425,231.754461,510.669604,NaN,NaN,6.189788,...,portwatch_pending,NaN,NaN,NaN,NaN,port_congestion_hours and chokepoint intensity...,0,NaN,NaN,0.5


### Source quality check

If `portwatch` is `pending_real_csv`, download a PortWatch chokepoint export into `data/raw/` and re-run this notebook so `chokepoint_vessel_count` / congestion fields are real vessel indicators.


In [3]:
meta_path = PATHS["raw"] / "collection_meta.json"
print(meta_path.read_text())
print("Columns:", list(panel.columns))
print("Date range:", panel["date"].min(), "->", panel["date"].max())
print("Non-null physical congestion:", panel["port_congestion_hours"].notna().sum())


{
  "failures": [
    "EIA unavailable or key missing",
    "PortWatch CSV not downloadable yet; physical IVs use observed market stress stand-ins documented in physical_iv_note (replace when PortWatch file is added to data/raw)"
  ],
  "sources_ok": [
    "FRED",
    "GDELT"
  ],
  "portwatch": "pending_real_csv"
}
Columns: ['date', 'wti_spot', 'brent_spot', 'natgas_hh', 'wheat', 'corn', 'soy', 'gdelt_tone', 'commodity_roll_yield', 'grain_calendar_spread', 'refining_crack_spread', 'source_notes', 'bottleneck_index_5d', 'port_congestion_hours', 'chokepoint_vessel_count', 'vessel_draught_anomaly', 'physical_iv_note', 'volatility_regime', 'omega_ratio_proxy', 'news_event_impact_score', 'finbert_risk_probability']
Date range: 2021-01-01 00:00:00 -> 2026-10-14 00:00:00
Non-null physical congestion: 2105
